# Spelling Unseen Akkadian Words in Cuneiform Signs

Reference solution. Three progressive submissions: (1) a stem+suffix heuristic built on the `attested` column, (2) a character-level encoder-decoder transformer trained from scratch on `train.csv`, (3) the transformer with the heuristic as fallback where decoding produced no sign. The final file is `working/submission.csv`; its score on the answer key is the published reference.

The data directory is discovered from the environment.

In [ ]:
import os, glob, csv, re, types
def find_data():
    for cand in [os.environ.get('DATA_DIR', ''), '/data', '/data/public', 'dataset/public', '../input', '/kaggle/input', '.']:
        if not cand:
            continue
        if os.path.exists(os.path.join(cand, 'train.csv')):
            return cand
        if os.path.isdir(cand):
            hits = sorted(glob.glob(os.path.join(cand, '**', 'train.csv'), recursive=True))
            if hits:
                return os.path.dirname(hits[0])
    raise FileNotFoundError('train.csv not found; set DATA_DIR')
DATA = find_data()
os.makedirs('working', exist_ok=True)
# the published reference configuration (about 5.7M parameters, fp32)
CFG = dict(data=DATA, out='working/submission_model.csv', d=256, heads=4, layers=3, epochs=3, batch=48, lr=4e-4,
           device='auto', max_train=250000, dev_frac=0.03, maxi=224, seed=0, mps_frac=0.6, budget_min=80,
           beam=4, save='working/model.pt')
a = types.SimpleNamespace(**CFG)
print(DATA, CFG)

## Submission 1: stem+suffix heuristic
Copies the nearest attested spelling of the lemma (longest shared normalized prefix) and syllabifies the rest of the word. It uses exactly the information in the `attested` column and is the baseline the model has to beat.

In [ ]:
def read_rows(p):
    with open(p, encoding='utf-8', newline='') as fh:
        return list(csv.DictReader(fh))
test_rows = read_rows(os.path.join(DATA, 'test.csv'))
SIGN = re.compile(r'\{[^}]*\}|[^{}\-.:\s]+')
def strip_len(n): return n.translate(str.maketrans('āēīūâêîû', 'aeiuaeiu'))
def syll(n):
    n = strip_len(n).lower().replace('ʾ', ''); V = set('aeiu'); out = []; i = 0
    while i < len(n):
        c = n[i]
        if c in V:
            out.append(c); i += 1
        elif i + 1 < len(n) and n[i + 1] in V:
            if i + 3 < len(n) and n[i + 2] not in V and n[i + 3] not in V:
                out.append(n[i:i + 3]); i += 3
            else:
                out.append(n[i:i + 2]); i += 2
        else:
            out.append(c); i += 1
    return '-'.join(out)
def att_pairs(s):
    return [tuple(part.split('=', 1)) for part in s.split(' ; ') if '=' in part]
def stem_suffix(norm, attested):
    n = strip_len(norm).lower(); best = None; bl = -1
    for tn, tf in att_pairs(attested):
        tn2 = strip_len(tn).lower(); l = 0
        while l < min(len(n), len(tn2)) and n[l] == tn2[l]:
            l += 1
        if l > bl:
            bl = l; best = (tn2, tf)
    if best is None or bl == 0:
        return syll(norm)
    toks = SIGN.findall(best[1])
    if toks and any(t.isupper() for t in toks if not t.startswith('{')):
        core = [t for t in toks if t.isupper() or t.startswith('{')]; rem = n[bl:]
        return '-'.join(core + ([syll(rem)] if rem else []))
    acc = 0; keep = []
    for t in toks:
        if t.startswith('{'):
            keep.append(t); continue
        tt = re.sub(r'[₀-₉]', '', t)
        if acc + len(tt) <= bl:
            keep.append(t); acc += len(tt)
        else:
            break
    rem = n[acc:]
    return '-'.join(keep + ([syll(rem)] if rem else []))
heur = {r['row_id']: stem_suffix(r['norm'], r['attested']) for r in test_rows}
with open('working/submission_heuristic.csv', 'w', encoding='utf-8', newline='') as fh:
    w = csv.writer(fh, lineterminator='\n'); w.writerow(['row_id', 'spelling'])
    for r in test_rows:
        w.writerow([r['row_id'], heur[r['row_id']]])
print('heuristic submission written:', len(heur), 'rows')

## Retrieval of attested spellings
Identical to `work/retrieval.py`. For each row it retrieves, from the training rows only, the lemma's most frequent form plus the lemma's forms with the longest shared normalized prefix, and forms of other lemmas with the same part of speech and the longest shared ending. A row never retrieves its own normalized form, and a training row never retrieves a form attested only in its own text, because evaluation texts are unseen.

In [ ]:
"""Per-row retrieval of attested spellings from the training rows (train-only statistics; nothing from
evaluation rows). For a row with normalized form n, lemma l, part of speech p and text d:
  - lemma neighbours: the lemma's most frequent training form, plus the lemma's forms whose folded
    normalized form shares the longest prefix with n (sorted-list neighbours), up to 4 in all;
  - ending neighbours: forms of OTHER lemmas with the same part of speech whose folded normalized form
    shares the longest suffix with n (2 to 5 characters), up to 2.
A form is never the row's own normalized form, and for a training row never a form attested only in
the row's own text (evaluation texts are unseen, so training must not see same-text spellings)."""
import bisect
from collections import defaultdict, Counter

FOLD = str.maketrans('āēīūâêîû', 'aeiuaeiu')
def fold(n): return n.translate(FOLD).replace('ʾ', '')

class Retriever:
    def __init__(self, rows, docmap):
        pair_ct = Counter(); pair_docs = defaultdict(set)
        for r in rows:
            k = (r['lemma'], r['norm'], r['spelling']); pair_ct[k] += 1
            ds = pair_docs[k]
            if len(ds) < 2: ds.add(docmap.get(r['row_id']))
        # per lemma: per norm, its best spelling (most frequent), count, docs (<=2 kept)
        best = {}
        for (lem, norm, sp), c in pair_ct.items():
            key = (lem, norm)
            if key not in best or c > best[key][1] or (c == best[key][1] and sp < best[key][0]):
                best[key] = (sp, c, pair_docs[(lem, norm, sp)])
        self.lem_norms = defaultdict(list)   # lemma -> sorted list of (folded, norm)
        self.lem_top = {}                    # lemma -> list of (count, norm, spelling, docs) desc
        tops = defaultdict(list)
        for (lem, norm), (sp, c, ds) in best.items():
            self.lem_norms[lem].append((fold(norm), norm)); tops[lem].append((c, norm, sp, ds))
        for lem in self.lem_norms: self.lem_norms[lem].sort()
        for lem, v in tops.items(): self.lem_top[lem] = sorted(v, key=lambda x: (-x[0], x[1]))[:6]
        self.best = best
        # endings: (pos, suffix) -> top forms by count (lemma, norm, spelling, docs)
        suf = defaultdict(list)
        for (lem, norm), (sp, c, ds) in best.items():
            f = fold(norm); pos = self._pos.get((lem, norm), '')
            for k in range(2, 6):
                if len(f) > k: suf[(pos, f[-k:])].append((c, lem, norm, sp, ds))
        self.suf = {k: sorted(v, key=lambda x: (-x[0], x[2]))[:12] for k, v in suf.items()}
    _pos = {}
    @classmethod
    def build(cls, rows, docmap):
        cls._pos = {}
        for r in rows: cls._pos.setdefault((r['lemma'], r['norm']), r['pos'])
        return cls(rows, docmap)
    @staticmethod
    def _ok(norm, ds, own_norm, own_doc):
        return norm != own_norm and not (own_doc is not None and ds == {own_doc})
    def lemma_pairs(self, r, own_doc=None, k=4):
        lem, norm = r['lemma'], r['norm']; out = []; seen = set()
        for c, n, sp, ds in self.lem_top.get(lem, []):
            if self._ok(n, ds, norm, own_doc):
                out.append((n, sp)); seen.add(n); break
        L = self.lem_norms.get(lem, [])
        if L:
            f = fold(norm); i = bisect.bisect_left(L, (f, norm))
            cand = []
            for j in range(max(0, i - 6), min(len(L), i + 6)):
                fj, nj = L[j]
                if nj in seen: continue
                sp, c, ds = self.best[(lem, nj)]
                if not self._ok(nj, ds, norm, own_doc): continue
                l = 0; m = min(len(f), len(fj))
                while l < m and f[l] == fj[l]: l += 1
                cand.append((-l, -c, nj, sp))
            cand.sort()
            for _, _, nj, sp in cand:
                if len(out) >= k: break
                out.append((nj, sp)); seen.add(nj)
        return out
    def ending_pairs(self, r, own_doc=None, k=2):
        f = fold(r['norm']); out = []
        for kk in range(5, 1, -1):
            if len(f) <= kk: continue
            for c, lem, n, sp, ds in self.suf.get((r['pos'], f[-kk:]), []):
                if lem == r['lemma'] or not self._ok(n, ds, r['norm'], own_doc): continue
                if any(n == x for x, _ in out): continue
                out.append((n, sp))
                if len(out) >= k: return out
        return out
    def text(self, r, own_doc=None):
        L = ' ; '.join(f'{n}={s}' for n, s in self.lemma_pairs(r, own_doc))
        E = ' ; '.join(f'{n}={s}' for n, s in self.ending_pairs(r, own_doc))
        return L, E


## Submission 2: character transformer
Identical to `work/reference.py` below its argument parsing. Its input is the row's part of speech, lemma, gloss and normalized form, the retrieved spellings, and the period and genre. The vocabulary is built from training rows only. A development split of whole training texts (from `train_docs.csv`), restricted to normalized forms unseen in the remaining texts, is scored with the task metric (greedy and beam) before the test set is decoded with beam search. A time budget is checked at epoch boundaries only.

In [ ]:
import contextlib, csv, math, os, random, re, sys, time
import torch, torch.nn as nn
random.seed(a.seed); torch.manual_seed(a.seed)
dev = a.device if a.device != 'auto' else ('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
def rd(p):
    with open(p, encoding='utf-8', newline='') as fh:
        return list(csv.DictReader(fh))
train = rd(os.path.join(a.data, 'train.csv')); test = rd(os.path.join(a.data, 'test.csv'))
print('train', len(train), 'test', len(test), 'device', dev, flush=True)
TOK = re.compile(r"\{[^}]*\}|[^{}\-.:\s]+")
def lev(p, q):
    if not p: return len(q)
    if not q: return len(p)
    prev = list(range(len(q) + 1))
    for i, x in enumerate(p, 1):
        cur = [i]
        for j, y in enumerate(q, 1): cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]
def row_score(pred, true):
    # the task metric: 0.5 * exact sign-sequence match + 0.5 * sign-level edit similarity
    p, t = TOK.findall(pred), TOK.findall(true)
    if not p and not t: return 1.0
    if not p or not t: return 0.0
    return 0.5 * (p == t) + 0.5 * (1 - lev(p, t) / max(len(p), len(t)))
docmap = {r['row_id']: r['doc'] for r in rd(os.path.join(a.data, 'train_docs.csv'))}
# development split built like the evaluation set: whole held-out docs, rows whose norm is unseen elsewhere
dev_rows = []
if a.dev_frac > 0:
    docs = sorted(set(docmap.values())); random.Random(a.seed).shuffle(docs)
    hold = set(docs[:int(len(docs) * a.dev_frac)])
    rest = [r for r in train if docmap[r['row_id']] not in hold]; seen = {r['norm'] for r in rest}
    dev_rows = [r for r in train if docmap[r['row_id']] in hold and r['norm'] not in seen][:4000]
    train = rest
    print('dev rows (held-out docs, unseen norms)', len(dev_rows), flush=True)
# retrieval index from the (non-dev) training rows only; a training row never retrieves its own form
# or a form attested only in its own text, matching evaluation rows (whose texts and forms are unseen)
R = Retriever.build(train, docmap)
def add_retrieval(rows, own_text):
    for r in rows:
        r['_L'], r['_E'] = R.text(r, docmap.get(r['row_id']) if own_text else None)
add_retrieval(train, True); add_retrieval(dev_rows, False); add_retrieval(test, False)
print('retrieval added', flush=True)
def enc_in(r):
    # metadata first, retrieved pairs last, so truncation at MAXI only ever drops the tail of the retrieved pairs
    return f"{r['pos']} | {r['period'][:14]} | {r['genre'][:14]} | {r['lemma']} | {r['gloss'][:20]} | {r['norm']} | L: {r['_L']} | E: {r['_E']}"
random.Random(a.seed).shuffle(train)
if a.max_train: train = train[:a.max_train]
# vocabulary from TRAINING rows only; characters never seen in training are dropped at encode time
chars = sorted({c for r in train for c in enc_in(r) + r['spelling']})
PAD, BOS, EOS = 0, 1, 2
stoi = {c: i + 3 for i, c in enumerate(chars)}; itos = {v: k for k, v in stoi.items()}; V = len(stoi) + 3
MAXI, MAXO = a.maxi, 40
def ids(s): return [stoi[c] for c in s if c in stoi]
def enc_x(r): return ids(enc_in(r))[:MAXI] or [PAD + 3]
def enc_y(r): return [BOS] + ids(r['spelling'])[:MAXO - 2] + [EOS]
TX = [enc_x(r) for r in train]; TY = [enc_y(r) for r in train]
FIXED = dev == 'mps'   # constant shapes on Apple MPS (its allocator fragments on varying shapes); bucketed elsewhere
def pad(seqs, cap):
    L = cap if FIXED else min(cap, -(-max(len(s) for s in seqs) // 16) * 16)
    out = torch.full((len(seqs), L), PAD, dtype=torch.long)
    for i, s in enumerate(seqs): out[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return out
class S2S(nn.Module):
    def __init__(s, d, h, L):
        super().__init__()
        s.emb = nn.Embedding(V, d); s.pi = nn.Embedding(MAXI, d); s.po = nn.Embedding(MAXO, d)
        s.tf = nn.Transformer(d, h, L, L, 4 * d, 0.1, batch_first=True)
        s.tf.encoder.enable_nested_tensor = False; s.tf.encoder.use_nested_tensor = False
        s.out = nn.Linear(d, V)
    def encode(s, x):
        xm = x == PAD
        return s.tf.encoder(s.emb(x) + s.pi(torch.arange(x.size(1), device=x.device)), src_key_padding_mask=xm), xm
    def decode(s, y, mem, xm):
        cm = nn.Transformer.generate_square_subsequent_mask(y.size(1), device=y.device)
        h = s.tf.decoder(s.emb(y) + s.po(torch.arange(y.size(1), device=y.device)), mem, tgt_mask=cm,
                         tgt_key_padding_mask=(y == PAD).float().masked_fill(y == PAD, float('-inf')), memory_key_padding_mask=xm.float().masked_fill(xm, float('-inf')))
        return s.out(h)
    def forward(s, x, y):
        mem, xm = s.encode(x)
        return s.decode(y, mem, xm)
m = S2S(a.d, a.heads, a.layers).to(dev)
opt = torch.optim.AdamW(m.parameters(), a.lr, weight_decay=0.01)
if dev == 'mps':
    try: torch.mps.set_per_process_memory_fraction(a.mps_frac)
    except Exception: pass
steps = a.epochs * math.ceil(len(train) / a.batch)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, a.lr, total_steps=steps, pct_start=0.1)
amp = contextlib.nullcontext   # full fp32 on every device, so a CUDA run reproduces the published MPS run's numerics as closely as possible
t0 = time.time(); stop = False; order = list(range(len(train)))
for ep in range(a.epochs):
    random.Random(a.seed + ep).shuffle(order); m.train(); tl = 0.0; nb = 0
    for i in range(0, len(order), a.batch):
        idx = order[i:i + a.batch]
        X = pad([TX[j] for j in idx], MAXI).to(dev); Y = pad([TY[j] for j in idx], MAXO).to(dev)
        with amp():
            logits = m(X, Y[:, :-1])
        loss = nn.functional.cross_entropy(logits.float().reshape(-1, V), Y[:, 1:].reshape(-1), ignore_index=PAD)
        opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step()
        tl += loss.item(); nb += 1
        if nb % 500 == 0: print(f'  ep{ep} b{nb} loss {tl / nb:.3f} {time.time() - t0:.0f}s', flush=True)
        if dev == 'mps' and nb % 200 == 0: torch.mps.empty_cache()
        if nb == 100 and ep == 0: print(f'  100 batches {time.time() - t0:.0f}s' + (f', mps {torch.mps.driver_allocated_memory() / 2**30:.2f} GiB' if dev == 'mps' else ''), flush=True)
    print(f'ep{ep} loss {tl / max(1, nb):.3f} {time.time() - t0:.0f}s', flush=True)
    if a.budget_min and ep + 1 < a.epochs and time.time() - t0 > a.budget_min * 60:
        print('time budget reached at an epoch boundary; training stopped', flush=True); break
@torch.no_grad()
def decode(rows, device):
    m.eval(); res = [''] * len(rows)
    order_ = sorted(range(len(rows)), key=lambda i: len(enc_x(rows[i])))
    B = 128 if device == 'mps' else 256
    for s0 in range(0, len(rows), B):
        bi = order_[s0:s0 + B]
        X = pad([enc_x(rows[i]) for i in bi], MAXI).to(device)
        with amp():
            mem, xm = m.encode(X)
        Y = torch.full((X.size(0), 1), BOS, dtype=torch.long, device=device); done = torch.zeros(X.size(0), dtype=torch.bool, device=device)
        for _ in range(MAXO - 1):
            with amp():
                nxt = m.decode(Y, mem, xm)[:, -1].argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)
            Y = torch.cat([Y, nxt[:, None]], 1); done |= nxt == EOS
            if bool(done.all()): break
        for k, row in zip(bi, Y.tolist()):
            out = []
            for t in row[1:]:
                if t in (EOS, PAD): break
                out.append(itos.get(t, ''))
            res[k] = ''.join(out)
        if device == 'mps': torch.mps.empty_cache()
    return res
@torch.no_grad()
def decode_beam(rows, device, beam):
    # batched beam search with the encoder memory computed once; final choice by length-normalised log-probability
    m.eval(); res = [''] * len(rows)
    order_ = sorted(range(len(rows)), key=lambda i: len(enc_x(rows[i])))
    B = 64 if device == 'mps' else 128
    for s0 in range(0, len(rows), B):
        bi = order_[s0:s0 + B]; n = len(bi)
        X = pad([enc_x(rows[i]) for i in bi], MAXI).to(device)
        with amp():
            mem, xm = m.encode(X)
        mem = mem.repeat_interleave(beam, 0); xm = xm.repeat_interleave(beam, 0)
        Y = torch.full((n * beam, 1), BOS, dtype=torch.long, device=device)
        score = torch.zeros(n, beam, device=device); score[:, 1:] = -1e9
        done = torch.zeros(n * beam, dtype=torch.bool, device=device); length = torch.zeros(n * beam, device=device)
        base = (torch.arange(n, device=device) * beam).unsqueeze(1)
        for _ in range(MAXO - 1):
            with amp():
                logits = m.decode(Y, mem, xm)[:, -1]
            logp = torch.log_softmax(logits.float(), -1)
            logp[done] = -1e9; logp[done, PAD] = 0.0
            top, idx = (score.view(-1, 1) + logp).view(n, -1).topk(beam, dim=1)
            sel = (base + idx // V).view(-1); tok = (idx % V).view(-1)
            prev_done = done[sel]
            length = length[sel] + (~prev_done).float()
            tok = torch.where(prev_done, torch.full_like(tok, PAD), tok)
            done = prev_done | (tok == EOS)
            Y = torch.cat([Y[sel], tok[:, None]], 1); score = top
            if bool(done.all()): break
        final = (score.view(-1) / length.clamp(min=1.0)).view(n, beam)
        pick = final.argmax(1)
        for k, (i, b) in enumerate(zip(bi, pick.tolist())):
            out = []
            for t_ in Y[k * beam + b, 1:].tolist():
                if t_ in (EOS, PAD): break
                out.append(itos.get(t_, ''))
            res[i] = ''.join(out)
        if device == 'mps': torch.mps.empty_cache()
    return res
if a.save:
    os.makedirs(os.path.dirname(a.save) or '.', exist_ok=True)
    torch.save({'state': m.state_dict(), 'stoi': stoi, 'cfg': vars(a)}, a.save)
ddev = dev
try:
    decode(test[:4], ddev)
except Exception as e:
    print('decode on', ddev, 'failed (', str(e)[:80], ') -> cpu', flush=True); ddev = 'cpu'; m = m.to('cpu')
dec = (lambda rows, device: decode_beam(rows, device, a.beam)) if a.beam > 1 else decode
if dev_rows:
    p = decode(dev_rows, ddev)
    print(f'dev score (greedy) {sum(row_score(x, r["spelling"]) for x, r in zip(p, dev_rows)) / len(dev_rows):.4f}', flush=True)
    p = dec(dev_rows, ddev)
    print(f'dev score {sum(row_score(x, r["spelling"]) for x, r in zip(p, dev_rows)) / len(dev_rows):.4f} exact {sum(TOK.findall(x) == TOK.findall(r["spelling"]) for x, r in zip(p, dev_rows)) / len(dev_rows):.4f} on {len(dev_rows)} rows', flush=True)
preds = dec(test, ddev)
os.makedirs(os.path.dirname(a.out) or '.', exist_ok=True)
with open(a.out, 'w', encoding='utf-8', newline='') as fh:
    w = csv.writer(fh, lineterminator='\n'); w.writerow(['row_id', 'spelling'])
    for r, p in zip(test, preds): w.writerow([r['row_id'], p])
print('wrote', a.out, f'{time.time() - t0:.0f}s', flush=True)


## Submission 3: transformer with heuristic fallback
Rows where greedy decoding produced no sign token take the heuristic's spelling.

In [ ]:
final = {}; n_fallback = 0
with open(CFG['out'], encoding='utf-8', newline='') as fh:
    for r in csv.DictReader(fh):
        if SIGN.findall(r['spelling']):
            final[r['row_id']] = r['spelling']
        else:
            final[r['row_id']] = heur[r['row_id']]; n_fallback += 1
with open('working/submission.csv', 'w', encoding='utf-8', newline='') as fh:
    w = csv.writer(fh, lineterminator='\n'); w.writerow(['row_id', 'spelling'])
    for r in test_rows:
        w.writerow([r['row_id'], final.get(r['row_id'], heur[r['row_id']])])
print('final submission written:', len(test_rows), 'rows; heuristic fallbacks used:', n_fallback + sum(1 for r in test_rows if r['row_id'] not in final))